# FarmResilience Analytics: Portfolio Dataset Preparation

## Objective

Prepare a clean, reusable analytical dataset for dashboard development using FAOSTAT agricultural production data for Kenya from 1961 to 2024.

## Scope

The analysis covers five selected crops:

- Maize (corn)
- Beans, dry
- Sorghum
- Millet
- Cassava, fresh

The dataset includes production, area harvested and yield.

## Data preparation principles

- Preserve the original observations and retain observation flags where available.
- Check for missing values, duplicate records and unexpected categories.
- Keep units explicit because production, harvested area and yield use different units.
- Avoid treating missing observations as zero.
- Preserve the distinction between reported values and estimated or imputed observations.

### Step 1 — Load and prepare the data

In [1]:
import pandas as pd
import zipfile
from pathlib import Path

# Locate the source file
zip_path = Path("Production_Crops_Livestock_E_Africa.zip")

if not zip_path.exists():
    raise FileNotFoundError(
        "Place Production_Crops_Livestock_E_Africa.zip "
        "in the notebook's working directory."
    )

# Load the numerical observations
with zipfile.ZipFile(zip_path, "r") as archive:
    with archive.open(
        "Production_Crops_Livestock_E_Africa_NOFLAG.csv"
    ) as file:
        df = pd.read_csv(file, low_memory=False)

# Convert annual columns into long format
year_cols = [
    col for col in df.columns
    if col.startswith("Y") and col[1:].isdigit()
]

id_cols = [col for col in df.columns if col not in year_cols]

df_long = df.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="Year",
    value_name="Value"
)

df_long["Year"] = (
    df_long["Year"]
    .str.replace("Y", "", regex=False)
    .astype(int)
)

df_long["Value"] = pd.to_numeric(
    df_long["Value"],
    errors="coerce"
)

# Select Kenya and the five crops
selected_crops = [
    "Maize (corn)",
    "Beans, dry",
    "Sorghum",
    "Millet",
    "Cassava, fresh"
]

selected_elements = [
    "Production",
    "Area harvested",
    "Yield"
]

df_crop = df_long[
    (df_long["Area"] == "Kenya")
    & (df_long["Item"].isin(selected_crops))
    & (df_long["Element"].isin(selected_elements))
].copy()

print("Dataset preparation complete.")
print("Rows:", len(df_crop))
print("Columns:", len(df_crop.columns))
print("Year range:", df_crop["Year"].min(), "-", df_crop["Year"].max())

display(df_crop.head())

Dataset preparation complete.
Rows: 960
Columns: 11
Year range: 1961 - 2024


,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Unit,Year,Value
6327,114,'404,Kenya,176,'01701,"Beans, dry",5312,Area harvested,ha,1961,115000.0
6328,114,'404,Kenya,176,'01701,"Beans, dry",5412,Yield,kg/ha,1961,478.3
6329,114,'404,Kenya,176,'01701,"Beans, dry",5510,Production,t,1961,55000.0
6348,114,'404,Kenya,125,'01520.01,"Cassava, fresh",5312,Area harvested,ha,1961,55000.0
6349,114,'404,Kenya,125,'01520.01,"Cassava, fresh",5412,Yield,kg/ha,1961,7636.4


### Step 1.2 — Merge the observation flags

In [2]:

# Load the original FAOSTAT dataset containing observation flags
with zipfile.ZipFile(zip_path, "r") as archive:
    with archive.open(
        "Production_Crops_Livestock_E_Africa.csv"
    ) as file:
        df_flag_source = pd.read_csv(file, low_memory=False)

# Identify annual flag columns such as Y1961F and Y2024F
flag_year_cols = [
    col for col in df_flag_source.columns
    if col.startswith("Y")
    and col.endswith("F")
    and col[1:-1].isdigit()
]

print("Flag columns identified:", len(flag_year_cols))

# Reshape the flag data from wide to long format
flag_keys = ["Area Code", "Item Code", "Element Code"]

flags_long = df_flag_source[
    flag_keys + flag_year_cols
].melt(
    id_vars=flag_keys,
    value_vars=flag_year_cols,
    var_name="Flag_Year",
    value_name="Observation_Flag"
)

# Extract the year from each flag-column name
flags_long["Year"] = (
    flags_long["Flag_Year"]
    .str.extract(r"Y(\d{4})F")[0]
    .astype(int)
)

# Clean the flag values
flags_long["Observation_Flag"] = (
    flags_long["Observation_Flag"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

# Keep only observations with a recorded flag
flags_long = flags_long.dropna(
    subset=["Observation_Flag"]
).copy()

# Merge flags into the selected crop dataset
merge_keys = [
    "Area Code",
    "Item Code",
    "Element Code",
    "Year"
]

df_crop = df_crop.merge(
    flags_long[merge_keys + ["Observation_Flag"]],
    on=merge_keys,
    how="left",
    validate="one_to_one"
)

print("\nFlag merge complete.")
print("Rows after merge:", len(df_crop))
print("Columns after merge:", len(df_crop.columns))

display(
    df_crop["Observation_Flag"]
    .value_counts(dropna=False)
    .rename_axis("Observation_Flag")
    .to_frame("Count")
)

display(
    df_crop[
        ["Item", "Element", "Year", "Value", "Observation_Flag"]
    ].head(10)
)


Flag columns identified: 64

Flag merge complete.
Rows after merge: 960
Columns after merge: 12


,Count
Observation_Flag,
A,684
E,248
X,28


,Item,Element,Year,Value,Observation_Flag
0,"Beans, dry",Area harvested,1961,115000.0,E
1,"Beans, dry",Yield,1961,478.3,E
2,"Beans, dry",Production,1961,55000.0,E
3,"Cassava, fresh",Area harvested,1961,55000.0,E
4,"Cassava, fresh",Yield,1961,7636.4,E
5,"Cassava, fresh",Production,1961,420000.0,E
6,Maize (corn),Area harvested,1961,750000.0,E
7,Maize (corn),Yield,1961,1253.3,E
8,Maize (corn),Production,1961,940000.0,E
9,Millet,Area harvested,1961,75000.0,E


## Step 1.3 — Final data-quality checks

In [3]:

# 1. Check dataset dimensions
print("DATASET DIMENSIONS")
print("Rows:", len(df_crop))
print("Columns:", len(df_crop.columns))
print("Year range:", df_crop["Year"].min(), "-", df_crop["Year"].max())

# 2. Check for duplicate crop-measure-year records
key_columns = ["Area Code", "Item Code", "Element Code", "Year"]

duplicate_count = df_crop.duplicated(
    subset=key_columns
).sum()

print("\nDUPLICATE CHECK")
print("Duplicate records:", duplicate_count)

# 3. Check missing values
print("\nMISSING VALUES")
display(
    df_crop.isna()
    .sum()
    .rename("Missing_Count")
    .to_frame()
)

# 4. Confirm expected crop and measure counts
print("\nCROP AND MEASURE COVERAGE")
print("Number of crops:", df_crop["Item"].nunique())
print("Crops:", sorted(df_crop["Item"].unique()))
print("Measures:", sorted(df_crop["Element"].unique()))

coverage = (
    df_crop.groupby(["Item", "Element"])["Year"]
    .agg(["count", "min", "max", "nunique"])
    .reset_index()
)

display(coverage)

# 5. Review observation flags
print("\nOBSERVATION FLAG DISTRIBUTION")
display(
    df_crop["Observation_Flag"]
    .value_counts(dropna=False)
    .rename_axis("Observation_Flag")
    .to_frame("Count")
)

# 6. Check for non-positive values
print("\nVALUE CHECK")
print("Zero or negative values:", (df_crop["Value"] <= 0).sum())
print("Non-numeric/missing values:", df_crop["Value"].isna().sum())


DATASET DIMENSIONS
Rows: 960
Columns: 12
Year range: 1961 - 2024

DUPLICATE CHECK
Duplicate records: 0

MISSING VALUES


,Missing_Count
Area Code,0
Area Code (M49),0
Area,0
Item Code,0
Item Code (CPC),0
Item,0
Element Code,0
Element,0
Unit,0
Year,0



CROP AND MEASURE COVERAGE
Number of crops: 5
Crops: ['Beans, dry', 'Cassava, fresh', 'Maize (corn)', 'Millet', 'Sorghum']
Measures: ['Area harvested', 'Production', 'Yield']


,Item,Element,count,min,max,nunique
0,"Beans, dry",Area harvested,64,1961,2024,64
1,"Beans, dry",Production,64,1961,2024,64
2,"Beans, dry",Yield,64,1961,2024,64
3,"Cassava, fresh",Area harvested,64,1961,2024,64
4,"Cassava, fresh",Production,64,1961,2024,64
5,"Cassava, fresh",Yield,64,1961,2024,64
6,Maize (corn),Area harvested,64,1961,2024,64
7,Maize (corn),Production,64,1961,2024,64
8,Maize (corn),Yield,64,1961,2024,64
9,Millet,Area harvested,64,1961,2024,64



OBSERVATION FLAG DISTRIBUTION


,Count
Observation_Flag,
A,684
E,248
X,28



VALUE CHECK
Zero or negative values: 0
Non-numeric/missing values: 0


### Step 1.4 — Prepare and export the final dataset
Now we'll create two files:
1. kenya_crop_trends_powerbi.csv — the detailed dataset for Power BI.
2. kenya_crop_summary_powerbi.csv — a summary of production, harvested area, and yield for 1961 versus 2024.

In [4]:

# Create an output folder for portfolio datasets
output_dir = Path("portfolio_outputs")
output_dir.mkdir(exist_ok=True)

# 1. Prepare the detailed analytical dataset
df_export = df_crop.copy()

# Standardize column names for easier use in Power BI
df_export.columns = [
    col.strip().replace(" ", "_").replace("(", "").replace(")", "")
    for col in df_export.columns
]

# Arrange records consistently
df_export = df_export.sort_values(
    ["Item", "Year", "Element"]
).reset_index(drop=True)

# Export the detailed dataset
detail_path = output_dir / "kenya_crop_trends_powerbi.csv"
df_export.to_csv(detail_path, index=False)

# 2. Create a 1961 vs 2024 comparison summary
endpoint_data = df_crop[
    df_crop["Year"].isin([1961, 2024])
].copy()

summary = endpoint_data.pivot_table(
    index=["Item", "Element", "Unit"],
    columns="Year",
    values="Value",
    aggfunc="first"
).reset_index()

summary.columns.name = None
summary = summary.rename(columns={
    1961: "Value_1961",
    2024: "Value_2024"
})

summary["Absolute_Change"] = (
    summary["Value_2024"] - summary["Value_1961"]
)

summary["Percentage_Change"] = (
    (summary["Value_2024"] - summary["Value_1961"])
    / summary["Value_1961"]
) * 100

summary_path = output_dir / "kenya_crop_summary_powerbi.csv"
summary.to_csv(summary_path, index=False)

# 3. Confirm exports
print("EXPORT COMPLETE")
print("Detailed dataset:", detail_path.resolve())
print("Detailed dataset rows:", len(df_export))
print("Summary dataset:", summary_path.resolve())
print("Summary rows:", len(summary))

print("\nDetailed dataset preview:")
display(df_export.head())

print("\nSummary dataset preview:")
display(summary.head(10))


EXPORT COMPLETE
Detailed dataset: C:\Users\User\Downloads\Agricultural resilience in Kenya\portfolio_outputs\kenya_crop_trends_powerbi.csv
Detailed dataset rows: 960
Summary dataset: C:\Users\User\Downloads\Agricultural resilience in Kenya\portfolio_outputs\kenya_crop_summary_powerbi.csv
Summary rows: 15

Detailed dataset preview:


,Area_Code,Area_Code_M49,Area,Item_Code,Item_Code_CPC,Item,Element_Code,Element,Unit,Year,Value,Observation_Flag
0,114,'404,Kenya,176,'01701,"Beans, dry",5312,Area harvested,ha,1961,115000.0,E
1,114,'404,Kenya,176,'01701,"Beans, dry",5510,Production,t,1961,55000.0,E
2,114,'404,Kenya,176,'01701,"Beans, dry",5412,Yield,kg/ha,1961,478.3,E
3,114,'404,Kenya,176,'01701,"Beans, dry",5312,Area harvested,ha,1962,115000.0,E
4,114,'404,Kenya,176,'01701,"Beans, dry",5510,Production,t,1962,55000.0,E



Summary dataset preview:


,Item,Element,Unit,Value_1961,Value_2024,Absolute_Change,Percentage_Change
0,"Beans, dry",Area harvested,ha,115000.0,1229882.00,1114882.00,969.462609
1,"Beans, dry",Production,t,55000.0,903850.98,848850.98,1543.365418
2,"Beans, dry",Yield,kg/ha,478.3,734.90,256.60,53.648338
3,"Cassava, fresh",Area harvested,ha,55000.0,81944.00,26944.00,48.989091
4,"Cassava, fresh",Production,t,420000.0,1207537.34,787537.34,187.508890
5,"Cassava, fresh",Yield,kg/ha,7636.4,14736.10,7099.70,92.971819
6,Maize (corn),Area harvested,ha,750000.0,2414536.00,1664536.00,221.938133
7,Maize (corn),Production,t,940000.0,4023000.00,3083000.00,327.978723
8,Maize (corn),Yield,kg/ha,1253.3,1666.20,412.90,32.945025
9,Millet,Area harvested,ha,75000.0,112093.00,37093.00,49.457333


### Step 1.5 — Verify the exported files
Before moving into Power BI, let's confirm that both CSV files exist, the detailed dataset has 960 rows, and the summary contains 15 records (5 crops × 3 measures).

In [5]:
# Verify that both exported CSV files can be read back
df_check = pd.read_csv(detail_path)
summary_check = pd.read_csv(summary_path)

print("FILE VERIFICATION")
print("Detailed file exists:", detail_path.exists())
print("Summary file exists:", summary_path.exists())

print("\nDETAILED DATASET")
print("Rows:", len(df_check))
print("Columns:", len(df_check.columns))
print("Observation_Flag retained:", "Observation_Flag" in df_check.columns)
print("Unique crops:", df_check["Item"].nunique())
print("Unique years:", df_check["Year"].nunique())

print("\nSUMMARY DATASET")
print("Rows:", len(summary_check))
print("Missing values:", summary_check.isna().sum().sum())

print("\nExported files:")
print(detail_path.resolve())
print(summary_path.resolve())


FILE VERIFICATION
Detailed file exists: True
Summary file exists: True

DETAILED DATASET
Rows: 960
Columns: 12
Observation_Flag retained: True
Unique crops: 5
Unique years: 64

SUMMARY DATASET
Rows: 15
Missing values: 0

Exported files:
C:\Users\User\Downloads\Agricultural resilience in Kenya\portfolio_outputs\kenya_crop_trends_powerbi.csv
C:\Users\User\Downloads\Agricultural resilience in Kenya\portfolio_outputs\kenya_crop_summary_powerbi.csv


Notebook 3 is complete. Both CSV files were exported and successfully verified. Your detailed dataset retains the observation flags, and the summary contains all 15 expected crop-measure comparisons.

Your project now has three completed notebooks:
- 01_faostat_data_audit.ipynb — data quality and source assessment.
- 02_crop_trends_and_resilience.ipynb — trends, productivity, and volatility analysis.
- 03_portfolio_dataset_preparation.ipynb — reusable, Power BI-ready datasets.
Next, we'll build the dashboard in Power BI Desktop, using the files you just exported.
### Step 4.1 — Import the detailed dataset into Power BI
### 1. Open Power BI Desktop

Start a new blank report.
### 2. Import the detailed CSV

Select Home → Get data → Text/CSV, then browse to:

C:\Users\User\Downloads\Agricultural resilience in Kenya\portfolio_outputs\kenya_crop_trends_powerbi.csv

### 3. Load the data

Review the preview and select Load. We can transform the data later if needed.

#### Step 4.2 — Check the imported column types
In Power BI's Data view, check that these fields have the correct types:
Column	Data type
Year	Whole number
Value	Decimal number
Item	Text
Element	Text
Unit	Text
Observation_Flag	Text

If Power BI assigned a different type, select the column and change it using Column tools → Data type.

Your next task: Import the detailed CSV and confirm that Power BI shows 960 rows.